# Unidade 3 - Bloco prático da Aula 02: o campeonato de recursos

Compara RandomizedSearchCV (todos os candidatos com os dados completos) e HalvingRandomSearchCV (funil com fator 3). Acompanhe o funil de eliminação rodada a rodada e o tempo total de cada estratégia.

In [2]:
import time
import numpy as np

from sklearn.datasets import make_classification
from sklearn.experimental import enable_halving_search_cv  # noqa

from sklearn.model_selection import (
    HalvingRandomSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

from scipy.stats import randint


# ============================================================
# 1. CRIANDO O DATASET
# ============================================================

print("=" * 70)
print("1. CRIANDO DATASET")
print("=" * 70)

X, y = make_classification(
    n_samples=6000,          # antes era 12000
    n_features=20,
    n_informative=8,
    weights=[0.8, 0.2],
    flip_y=0.02,
    random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print(f"Treino = {len(X_tr)} exemplos")
print(f"Teste  = {len(X_te)} exemplos")


# ============================================================
# 2. VALIDAÇÃO CRUZADA
# ============================================================

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


# ============================================================
# 3. ESPAÇO DE HIPERPARÂMETROS
# ============================================================

dist = {
    "n_estimators": randint(50, 200),

    "max_depth": [
        4,
        6,
        8,
        12,
        None
    ],

    "min_samples_leaf": randint(1, 10),

    "max_features": [
        "sqrt",
        "log2"
    ]
}


# ============================================================
# 4. MODELO
# ============================================================

# IMPORTANTE
#
# O RandomForest fica com apenas 1 núcleo.
# A busca é quem distribui os experimentos pela CPU.
#
# Isso evita paralelismo dentro de paralelismo.

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)


# ============================================================
# 5. RANDOMIZED SEARCH
# ============================================================

print("\n")
print("=" * 70)
print("2. RANDOMIZED SEARCH")
print("=" * 70)

print("Testando 20 configurações usando TODOS os dados de treino")
print()

rs = RandomizedSearchCV(
    estimator=rf,
    param_distributions=dist,

    n_iter=20,               # antes era 40

    cv=cv,
    scoring="f1",

    random_state=42,

    n_jobs=-1,

    # Mostra o andamento
    verbose=2
)


inicio = time.time()

rs.fit(
    X_tr,
    y_tr
)

tempo_rs = time.time() - inicio


print("\nRandomized Search terminou!")

print(
    f"Melhor F1 de validação = "
    f"{rs.best_score_:.4f}"
)

print(
    f"Melhores parâmetros = "
    f"{rs.best_params_}"
)

print(
    f"Tempo = "
    f"{tempo_rs:.1f} segundos"
)


# ============================================================
# 6. HALVING RANDOM SEARCH
# ============================================================

print("\n")
print("=" * 70)
print("3. HALVING RANDOM SEARCH")
print("=" * 70)

print(
    "Começa com vários candidatos usando poucos dados"
)

print(
    "Os piores são eliminados a cada rodada"
)

print(
    "Os melhores recebem progressivamente mais dados"
)

print()


hs = HalvingRandomSearchCV(
    estimator=rf,

    param_distributions=dist,

    n_candidates=20,

    # Recurso que aumenta a cada rodada
    resource="n_samples",

    # Primeira rodada usa poucos exemplos
    min_resources=500,

    # Aproximadamente 1/3 continua
    factor=3,

    cv=cv,
    scoring="f1",

    random_state=42,

    n_jobs=-1,

    # Mostra cada rodada
    verbose=2
)


inicio = time.time()

hs.fit(
    X_tr,
    y_tr
)

tempo_hs = time.time() - inicio


print("\nHalving Search terminou!")

print(
    f"Melhor F1 de validação = "
    f"{hs.best_score_:.4f}"
)

print(
    f"Melhores parâmetros = "
    f"{hs.best_params_}"
)

print(
    f"Tempo = "
    f"{tempo_hs:.1f} segundos"
)


# ============================================================
# 7. TESTE FINAL
# ============================================================

print("\n")
print("=" * 70)
print("4. TESTE FINAL")
print("=" * 70)


pred_rs = rs.predict(X_te)

pred_hs = hs.predict(X_te)


f1_rs = f1_score(
    y_te,
    pred_rs
)

f1_hs = f1_score(
    y_te,
    pred_hs
)


print(
    f"Randomized Search"
    f"\nCV    = {rs.best_score_:.4f}"
    f"\nTeste = {f1_rs:.4f}"
    f"\nTempo = {tempo_rs:.1f}s"
)


print()


print(
    f"Halving Random Search"
    f"\nCV    = {hs.best_score_:.4f}"
    f"\nTeste = {f1_hs:.4f}"
    f"\nTempo = {tempo_hs:.1f}s"
)


# ============================================================
# 8. COMPARAÇÃO
# ============================================================

print("\n")
print("=" * 70)
print("5. COMPARAÇÃO")
print("=" * 70)


if tempo_hs < tempo_rs:

    ganho = tempo_rs / tempo_hs

    print(
        f"Halving foi aproximadamente "
        f"{ganho:.2f}x mais rápido"
    )

else:

    ganho = tempo_hs / tempo_rs

    print(
        f"Randomized Search foi aproximadamente "
        f"{ganho:.2f}x mais rápido"
    )


print(
    f"\nDiferença de F1 no teste = "
    f"{abs(f1_rs - f1_hs):.4f}"
)

1. CRIANDO DATASET
Treino = 4500 exemplos
Teste  = 1500 exemplos


2. RANDOMIZED SEARCH
Testando 20 configurações usando TODOS os dados de treino

Fitting 3 folds for each of 20 candidates, totalling 60 fits

Randomized Search terminou!
Melhor F1 de validação = 0.6714
Melhores parâmetros = {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 107}
Tempo = 87.6 segundos


3. HALVING RANDOM SEARCH
Começa com vários candidatos usando poucos dados
Os piores são eliminados a cada rodada
Os melhores recebem progressivamente mais dados

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 500
max_resources_: 4500
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 20
n_resources: 500
Fitting 3 folds for each of 20 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 7
n_resources: 1500
Fitting 3 folds for each of 7 candidates, totalling 21 fits
----------
iter: 2
n_candidates: 3
n_resources: 4500
Fitting 3 